## ***Install and Import Libraries***

In [ ]:
!pip install -q statsmodels

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
import sys
import json
import urllib.request
from IPython.display import display, Image

# Statistical Tests
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu, ttest_ind, beta

# Proportion Inference & Confidence Intervals
from statsmodels.stats.proportion import proportions_ztest, proportion_confint, confint_proportions_2indep, proportion_effectsize

# Power Analysis & Multiple Testing
from statsmodels.stats.power import NormalIndPower, zt_ind_solve_power
from statsmodels.stats.multitest import multipletests

## ***Config***

In [ ]:
# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Configure plotting
# The multi-panel figures below are assembled one subplot per cell, so the inline
# backend is disabled to stop it closing the figure between cells
import matplotlib
matplotlib.use('agg')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (20, 15)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

In [ ]:
# Experiment configuration
ALPHA = 0.05
POWER_TARGET = 0.80
N_BOOTSTRAP = 10000
N_SIMULATIONS = 10000
CONTROL = 'gate_30'
TREATMENT = 'gate_40'

print(f"Significance level (alpha): {ALPHA}")
print(f"Target power: {POWER_TARGET}")
print(f"Control group: {CONTROL}")
print(f"Treatment group: {TREATMENT}")

In [ ]:
# Metric definitions
PRIMARY_METRIC = 'retention_7'
SECONDARY_METRIC = 'retention_1'
GUARDRAIL_METRIC = 'sum_gamerounds'

print(f"Primary metric: {PRIMARY_METRIC}")
print(f"Secondary metric: {SECONDARY_METRIC}")
print(f"Guardrail metric: {GUARDRAIL_METRIC}")

## ***Data Loading***

In [ ]:
# Load the dataset
DATA_URL = "https://raw.githubusercontent.com/ryanschaub/Mobile-Games-A-B-Testing-with-Cookie-Cats/master/cookie_cats.csv"
DATA_PATH = "data/cookie_cats.csv"

if not os.path.exists(DATA_PATH):
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print(f"Downloaded dataset to: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset from: {DATA_PATH}")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Display first few rows
print(df.head())

In [ ]:
# Check data types
print(df.dtypes)

In [ ]:
# Check for missing values
missing_count = df.isnull().sum().sum()
if missing_count == 0:
    print("No missing values found")
else:
    print(df.isnull().sum())

In [ ]:
# Check for duplicate users
duplicate_count = df['userid'].duplicated().sum()
print(f"Duplicate user IDs: {duplicate_count}")
if duplicate_count > 0:
    df = df.drop_duplicates(subset='userid')
    print(f"Removed {duplicate_count} duplicate users")

In [ ]:
# Dataset shape
print(f"Dataset Shape")
print(f"Total samples: {df.shape[0]}")
print(f"Total features: {df.shape[1]}")

## ***Data Preprocessing***

In [ ]:
# Convert boolean retention flags to integers
df['retention_1'] = df['retention_1'].astype(int)
df['retention_7'] = df['retention_7'].astype(int)
print("Converted retention flags to integer")

In [ ]:
# Check group assignment counts
version_counts = df['version'].value_counts()
print(version_counts)
print(f"Group Allocation")
print(f"{CONTROL} (control):   {version_counts[CONTROL]} ({version_counts[CONTROL]/len(df)*100:.3f}%)")
print(f"{TREATMENT} (treatment): {version_counts[TREATMENT]} ({version_counts[TREATMENT]/len(df)*100:.3f}%)")

In [ ]:
# Split into control and treatment frames
control = df[df['version'] == CONTROL]
treatment = df[df['version'] == TREATMENT]

n_control = len(control)
n_treatment = len(treatment)

print(f"Control samples: {n_control}")
print(f"Treatment samples: {n_treatment}")

In [ ]:
# Display descriptive statistics
print(df.describe())

## ***Exploratory Data Analysis***

In [ ]:
# Day-1 retention by group
print("Day-1 Retention by Group")
retention_1_table = pd.crosstab(df['version'], df['retention_1'], margins=True)
print(retention_1_table)

In [ ]:
# Day-7 retention by group
print("Day-7 Retention by Group")
retention_7_table = pd.crosstab(df['version'], df['retention_7'], margins=True)
print(retention_7_table)

In [ ]:
# Retention rates by group
print("Retention Rates by Group")
retention_rates = df.groupby('version')[['retention_1', 'retention_7']].mean() * 100
print(retention_rates.round(4))

In [ ]:
# Game rounds distribution by group
print("Game Rounds Played by Group")
print(df.groupby('version')['sum_gamerounds'].describe().round(4))

In [ ]:
# Players who never played a round
zero_rounds = (df['sum_gamerounds'] == 0).sum()
print(f"Players with 0 game rounds: {zero_rounds} ({zero_rounds/len(df)*100:.2f}%)")
print(df[df['sum_gamerounds'] == 0]['version'].value_counts())

In [ ]:
# Retention funnel
print("Retention Funnel")
both_retained = ((df['retention_1'] == 1) & (df['retention_7'] == 1)).sum()
only_day1 = ((df['retention_1'] == 1) & (df['retention_7'] == 0)).sum()
only_day7 = ((df['retention_1'] == 0) & (df['retention_7'] == 1)).sum()
neither = ((df['retention_1'] == 0) & (df['retention_7'] == 0)).sum()

print(f"Retained on both day 1 and day 7: {both_retained} ({both_retained/len(df)*100:.2f}%)")
print(f"Retained day 1 only:              {only_day1} ({only_day1/len(df)*100:.2f}%)")
print(f"Retained day 7 only:              {only_day7} ({only_day7/len(df)*100:.2f}%)")
print(f"Retained on neither day:          {neither} ({neither/len(df)*100:.2f}%)")

In [ ]:
# Correlation between the two retention metrics
retention_corr = df['retention_1'].corr(df['retention_7'])
print(f"Correlation between day-1 and day-7 retention: {retention_corr:.4f}")

In [ ]:
# Create comprehensive EDA visualizations
print("Creating EDA Visualizations")
fig = plt.figure(figsize=(20, 15))

In [ ]:
# Group Allocation
ax1 = plt.subplot(3, 4, 1)
version_counts.plot(kind='bar', ax=ax1, color=['skyblue', 'salmon'], edgecolor='black')
ax1.set_title('Group Allocation', fontweight='bold', fontsize=12)
ax1.set_xlabel('Version')
ax1.set_ylabel('Number of Players')
ax1.set_xticklabels([CONTROL, TREATMENT], rotation=0)

In [ ]:
# Day-1 Retention Rate
ax2 = plt.subplot(3, 4, 2)
retention_rates['retention_1'].plot(kind='bar', ax=ax2, color=['skyblue', 'salmon'], edgecolor='black')
ax2.set_title('Day-1 Retention Rate', fontweight='bold', fontsize=12)
ax2.set_xlabel('Version')
ax2.set_ylabel('Retention Rate (%)')
ax2.set_xticklabels([CONTROL, TREATMENT], rotation=0)
ax2.set_ylim(40, 46)
for i, v in enumerate(retention_rates['retention_1']):
    ax2.text(i, v + 0.1, f'{v:.2f}%', ha='center', fontweight='bold', fontsize=10)

In [ ]:
# Day-7 Retention Rate
ax3 = plt.subplot(3, 4, 3)
retention_rates['retention_7'].plot(kind='bar', ax=ax3, color=['skyblue', 'salmon'], edgecolor='black')
ax3.set_title('Day-7 Retention Rate', fontweight='bold', fontsize=12)
ax3.set_xlabel('Version')
ax3.set_ylabel('Retention Rate (%)')
ax3.set_xticklabels([CONTROL, TREATMENT], rotation=0)
ax3.set_ylim(16, 20)
for i, v in enumerate(retention_rates['retention_7']):
    ax3.text(i, v + 0.08, f'{v:.2f}%', ha='center', fontweight='bold', fontsize=10)

In [ ]:
# Game Rounds Distribution (log scale)
ax4 = plt.subplot(3, 4, 4)
ax4.hist(np.log1p(control['sum_gamerounds']), bins=50, alpha=0.5, label=CONTROL, color='blue')
ax4.hist(np.log1p(treatment['sum_gamerounds']), bins=50, alpha=0.5, label=TREATMENT, color='red')
ax4.set_title('Game Rounds Distribution (log1p)', fontweight='bold', fontsize=12)
ax4.set_xlabel('log1p(Game Rounds)')
ax4.set_ylabel('Frequency')
ax4.legend()

In [ ]:
# Game Rounds Boxplot
ax5 = plt.subplot(3, 4, 5)
df.boxplot(column='sum_gamerounds', by='version', ax=ax5)
ax5.set_title('Game Rounds by Version', fontweight='bold', fontsize=12)
ax5.set_xlabel('Version')
ax5.set_ylabel('Game Rounds')
ax5.set_yscale('log')
plt.suptitle('')

In [ ]:
# Retention Funnel
ax6 = plt.subplot(3, 4, 6)
funnel_labels = ['Both Days', 'Day 1 Only', 'Day 7 Only', 'Neither']
funnel_values = [both_retained, only_day1, only_day7, neither]
ax6.bar(funnel_labels, funnel_values, color=['seagreen', 'skyblue', 'gold', 'lightcoral'], edgecolor='black')
ax6.set_title('Retention Funnel', fontweight='bold', fontsize=12)
ax6.set_xlabel('Retention Pattern')
ax6.set_ylabel('Number of Players')
ax6.set_xticklabels(funnel_labels, rotation=45, ha='right')

In [ ]:
# Retention Rate by Game Rounds Bucket
ax7 = plt.subplot(3, 4, 7)
rounds_bucket = pd.cut(df['sum_gamerounds'], bins=[-1, 0, 5, 20, 50, 100, 1000000],
                       labels=['0', '1-5', '6-20', '21-50', '51-100', '100+'])
bucket_retention = df.groupby(rounds_bucket, observed=True)['retention_7'].mean() * 100
bucket_retention.plot(kind='bar', ax=ax7, color='mediumpurple', edgecolor='black')
ax7.set_title('Day-7 Retention by Rounds Played', fontweight='bold', fontsize=12)
ax7.set_xlabel('Game Rounds Bucket')
ax7.set_ylabel('Day-7 Retention (%)')
ax7.set_xticklabels(bucket_retention.index, rotation=45, ha='right')

In [ ]:
# Player Volume by Game Rounds Bucket
ax8 = plt.subplot(3, 4, 8)
bucket_counts = rounds_bucket.value_counts().sort_index()
bucket_counts.plot(kind='bar', ax=ax8, color='steelblue', edgecolor='black')
ax8.set_title('Player Volume by Rounds Played', fontweight='bold', fontsize=12)
ax8.set_xlabel('Game Rounds Bucket')
ax8.set_ylabel('Number of Players')
ax8.set_xticklabels(bucket_counts.index, rotation=45, ha='right')

In [ ]:
# Cumulative Game Rounds Distribution
ax9 = plt.subplot(3, 4, 9)
ax9.plot(np.sort(control['sum_gamerounds']), np.linspace(0, 1, n_control), label=CONTROL, color='blue', lw=2)
ax9.plot(np.sort(treatment['sum_gamerounds']), np.linspace(0, 1, n_treatment), label=TREATMENT, color='red', lw=2)
ax9.set_title('Cumulative Distribution of Game Rounds', fontweight='bold', fontsize=12)
ax9.set_xlabel('Game Rounds')
ax9.set_ylabel('Cumulative Proportion')
ax9.set_xscale('symlog')
ax9.legend()

In [ ]:
# Retention Rate Difference
ax10 = plt.subplot(3, 4, 10)
diff_labels = ['Day-1 Retention', 'Day-7 Retention']
diff_values = [retention_rates.loc[TREATMENT, 'retention_1'] - retention_rates.loc[CONTROL, 'retention_1'],
               retention_rates.loc[TREATMENT, 'retention_7'] - retention_rates.loc[CONTROL, 'retention_7']]
ax10.bar(diff_labels, diff_values, color=['lightcoral', 'indianred'], edgecolor='black')
ax10.axhline(0, color='black', lw=1)
ax10.set_title('Treatment Effect (pp)', fontweight='bold', fontsize=12)
ax10.set_ylabel('Difference (percentage points)')
for i, v in enumerate(diff_values):
    ax10.text(i, v - 0.05, f'{v:+.3f}pp', ha='center', fontweight='bold', fontsize=10)

In [ ]:
# Median Game Rounds by Group
ax11 = plt.subplot(3, 4, 11)
median_rounds = df.groupby('version')['sum_gamerounds'].median()
median_rounds.plot(kind='bar', ax=ax11, color=['skyblue', 'salmon'], edgecolor='black')
ax11.set_title('Median Game Rounds', fontweight='bold', fontsize=12)
ax11.set_xlabel('Version')
ax11.set_ylabel('Median Rounds')
ax11.set_xticklabels([CONTROL, TREATMENT], rotation=0)
for i, v in enumerate(median_rounds):
    ax11.text(i, v + 0.2, f'{v:.0f}', ha='center', fontweight='bold', fontsize=10)

In [ ]:
# Correlation Heatmap
ax12 = plt.subplot(3, 4, 12)
numerical_cols = ['sum_gamerounds', 'retention_1', 'retention_7']
correlation_matrix = df[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', ax=ax12,
            cbar_kws={'label': 'Correlation'}, linewidths=0.5)
ax12.set_title('Correlation Heatmap', fontweight='bold', fontsize=12)

In [ ]:
plt.tight_layout()
plt.savefig('outputs/eda_comprehensive_analysis.png', dpi=300, bbox_inches='tight')
plt.close()
print("EDA visualizations saved as 'outputs/eda_comprehensive_analysis.png'")
display(Image(filename='outputs/eda_comprehensive_analysis.png', width=1100))

## ***Experiment Validity Checks***

### ***Sample Ratio Mismatch***

In [ ]:
# Sample ratio mismatch test
expected_ratio = 0.5
observed_control_share = n_control / (n_control + n_treatment)

srm_chi2, srm_p = stats.chisquare(f_obs=[n_control, n_treatment],
                                  f_exp=[(n_control + n_treatment) * expected_ratio,
                                         (n_control + n_treatment) * (1 - expected_ratio)])

print(f"Sample Ratio Mismatch Check")
print(f"Expected allocation: {expected_ratio:.4f} / {1-expected_ratio:.4f}")
print(f"Observed allocation: {observed_control_share:.4f} / {1-observed_control_share:.4f}")
print(f"Chi-square statistic: {srm_chi2:.4f}")
print(f"P-value: {srm_p:.6f}")

In [ ]:
# Evaluate against standard SRM alerting thresholds
SRM_THRESHOLD_STRICT = 0.0005
SRM_THRESHOLD_DEFAULT = 0.001

print(f"SRM Verdict")
print(f"P-value vs strict threshold ({SRM_THRESHOLD_STRICT}):  {'ALERT' if srm_p < SRM_THRESHOLD_STRICT else 'PASS'}")
print(f"P-value vs default threshold ({SRM_THRESHOLD_DEFAULT}): {'ALERT' if srm_p < SRM_THRESHOLD_DEFAULT else 'PASS'}")
print(f"Absolute imbalance: {abs(n_control - n_treatment)} players ({abs(observed_control_share - expected_ratio)*100:.4f}pp from parity)")

### ***Outlier Detection***

In [ ]:
# Identify extreme values in the guardrail metric
rounds_q99 = df['sum_gamerounds'].quantile(0.99)
rounds_max = df['sum_gamerounds'].max()

print(f"Game Rounds Outlier Check")
print(f"99th percentile: {rounds_q99:.1f}")
print(f"Maximum value:   {rounds_max}")
print(f"Ratio of max to 99th percentile: {rounds_max/rounds_q99:.1f}x")

In [ ]:
# Inspect records above a plausible engagement ceiling
OUTLIER_THRESHOLD = 10000
outliers = df[df['sum_gamerounds'] > OUTLIER_THRESHOLD]
print(f"Records above {OUTLIER_THRESHOLD} rounds: {len(outliers)}")
print(outliers)

In [ ]:
# Build an outlier-excluded frame for guardrail robustness checks
df_no_outlier = df[df['sum_gamerounds'] <= OUTLIER_THRESHOLD]
control_no_outlier = df_no_outlier[df_no_outlier['version'] == CONTROL]
treatment_no_outlier = df_no_outlier[df_no_outlier['version'] == TREATMENT]

print(f"Rows retained: {len(df_no_outlier)} of {len(df)}")
print(f"Rows excluded: {len(df) - len(df_no_outlier)}")

### ***Power Analysis***

In [ ]:
# Baseline conversion rates from the control group
baseline_r1 = control['retention_1'].mean()
baseline_r7 = control['retention_7'].mean()

print(f"Baseline Rates (control group)")
print(f"Day-1 retention: {baseline_r1*100:.4f}%")
print(f"Day-7 retention: {baseline_r7*100:.4f}%")

In [ ]:
# Minimum detectable effect at the achieved sample size
power_analysis = NormalIndPower()

def compute_mde(baseline, n1, n2, alpha=ALPHA, power=POWER_TARGET):
    """Return the smallest absolute lift detectable at the given sample sizes."""
    effect_size = power_analysis.solve_power(effect_size=None, nobs1=n1, ratio=n2/n1,
                                            alpha=alpha, power=power)
    lo, hi = 0.0, 1.0 - baseline
    for _ in range(200):
        mid = (lo + hi) / 2
        if proportion_effectsize(baseline + mid, baseline) < effect_size:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

In [ ]:
mde_r1 = compute_mde(baseline_r1, n_control, n_treatment)
mde_r7 = compute_mde(baseline_r7, n_control, n_treatment)

print(f"Minimum Detectable Effect at n={n_control}/{n_treatment}, alpha={ALPHA}, power={POWER_TARGET}")
print(f"Day-1 retention: {mde_r1*100:.4f}pp absolute ({mde_r1/baseline_r1*100:.3f}% relative)")
print(f"Day-7 retention: {mde_r7*100:.4f}pp absolute ({mde_r7/baseline_r7*100:.3f}% relative)")

In [ ]:
# Sample size required per arm across a range of relative lifts
print(f"Sample Size Requirements per Arm (day-7 retention, alpha={ALPHA}, power={POWER_TARGET})")
sample_size_rows = []
for rel_lift in [0.01, 0.02, 0.03, 0.05, 0.075, 0.10]:
    effect_size = proportion_effectsize(baseline_r7 * (1 + rel_lift), baseline_r7)
    n_required = power_analysis.solve_power(effect_size=effect_size, nobs1=None, ratio=1.0,
                                           alpha=ALPHA, power=POWER_TARGET)
    sample_size_rows.append({'relative_lift': f"{rel_lift*100:.1f}%",
                             'absolute_lift_pp': round(baseline_r7 * rel_lift * 100, 4),
                             'n_per_arm': int(np.ceil(n_required))})

sample_size_df = pd.DataFrame(sample_size_rows)
print(sample_size_df.to_string(index=False))

## ***Primary Metric Analysis***

In [ ]:
# Reusable metric analysis routine
def analyze_proportion_metric(metric, control_frame, treatment_frame):
    """Run the full inference suite for one binary metric."""
    c, t = control_frame[metric], treatment_frame[metric]
    nc, nt = len(c), len(t)
    xc, xt = int(c.sum()), int(t.sum())
    pc, pt = xc / nc, xt / nt

    z_stat, z_p = proportions_ztest([xt, xc], [nt, nc])
    table = np.array([[xt, nt - xt], [xc, nc - xc]])
    chi2_stat, chi2_p, dof, expected = chi2_contingency(table, correction=False)
    odds_ratio, fisher_p = fisher_exact(table)
    ci_low, ci_high = confint_proportions_2indep(xt, nt, xc, nc, method='wald')
    effect_size = proportion_effectsize(pt, pc)
    achieved_power = power_analysis.power(effect_size=abs(effect_size), nobs1=nc,
                                          ratio=nt/nc, alpha=ALPHA)

    return {
        'metric': metric,
        'n_control': nc, 'n_treatment': nt,
        'x_control': xc, 'x_treatment': xt,
        'rate_control': pc, 'rate_treatment': pt,
        'absolute_lift': pt - pc,
        'relative_lift': pt / pc - 1,
        'z_statistic': z_stat, 'z_pvalue': z_p,
        'chi2_statistic': chi2_stat, 'chi2_pvalue': chi2_p,
        'odds_ratio': odds_ratio, 'fisher_pvalue': fisher_p,
        'ci_low': ci_low, 'ci_high': ci_high,
        'cohens_h': effect_size, 'achieved_power': achieved_power,
    }

In [ ]:
# Dictionary to store metric results
metric_results = {}

### ***Day-7 Retention***

In [ ]:
r7 = analyze_proportion_metric('retention_7', control, treatment)
metric_results['retention_7'] = r7

In [ ]:
print(f"Day-7 Retention")
print(f"{CONTROL}:   {r7['x_control']} / {r7['n_control']} = {r7['rate_control']*100:.4f}%")
print(f"{TREATMENT}: {r7['x_treatment']} / {r7['n_treatment']} = {r7['rate_treatment']*100:.4f}%")

In [ ]:
print(f"Effect Size")
print(f"Absolute lift: {r7['absolute_lift']*100:+.4f}pp")
print(f"Relative lift: {r7['relative_lift']*100:+.4f}%")
print(f"95% CI on absolute lift: [{r7['ci_low']*100:+.4f}pp, {r7['ci_high']*100:+.4f}pp]")

In [ ]:
print(f"Hypothesis Tests")
print(f"Two-proportion z-test: z = {r7['z_statistic']:.4f}, p = {r7['z_pvalue']:.6f}")
print(f"Chi-square test:       chi2 = {r7['chi2_statistic']:.4f}, p = {r7['chi2_pvalue']:.6f}")
print(f"Fisher exact test:     OR = {r7['odds_ratio']:.4f}, p = {r7['fisher_pvalue']:.6f}")

In [ ]:
print(f"Power")
print(f"Cohen's h: {r7['cohens_h']:.6f}")
print(f"Achieved power at observed effect: {r7['achieved_power']:.4f}")
print(f"Significant at alpha={ALPHA}: {r7['z_pvalue'] < ALPHA}")

### ***Day-1 Retention***

In [ ]:
r1 = analyze_proportion_metric('retention_1', control, treatment)
metric_results['retention_1'] = r1

In [ ]:
print(f"Day-1 Retention")
print(f"{CONTROL}:   {r1['x_control']} / {r1['n_control']} = {r1['rate_control']*100:.4f}%")
print(f"{TREATMENT}: {r1['x_treatment']} / {r1['n_treatment']} = {r1['rate_treatment']*100:.4f}%")

In [ ]:
print(f"Effect Size")
print(f"Absolute lift: {r1['absolute_lift']*100:+.4f}pp")
print(f"Relative lift: {r1['relative_lift']*100:+.4f}%")
print(f"95% CI on absolute lift: [{r1['ci_low']*100:+.4f}pp, {r1['ci_high']*100:+.4f}pp]")

In [ ]:
print(f"Hypothesis Tests")
print(f"Two-proportion z-test: z = {r1['z_statistic']:.4f}, p = {r1['z_pvalue']:.6f}")
print(f"Chi-square test:       chi2 = {r1['chi2_statistic']:.4f}, p = {r1['chi2_pvalue']:.6f}")
print(f"Fisher exact test:     OR = {r1['odds_ratio']:.4f}, p = {r1['fisher_pvalue']:.6f}")

In [ ]:
print(f"Power")
print(f"Cohen's h: {r1['cohens_h']:.6f}")
print(f"Achieved power at observed effect: {r1['achieved_power']:.4f}")
print(f"Significant at alpha={ALPHA}: {r1['z_pvalue'] < ALPHA}")

In [ ]:
# Sample size that would have been needed to detect the observed day-1 effect
n_needed_r1 = power_analysis.solve_power(effect_size=abs(r1['cohens_h']), nobs1=None,
                                        ratio=1.0, alpha=ALPHA, power=POWER_TARGET)
print(f"Observed day-1 effect: {r1['absolute_lift']*100:+.4f}pp")
print(f"Sample size per arm needed for {POWER_TARGET:.0%} power at that effect: {int(np.ceil(n_needed_r1))}")
print(f"Actual sample size per arm: {min(n_control, n_treatment)}")
print(f"Shortfall factor: {n_needed_r1/min(n_control, n_treatment):.2f}x")

## ***Guardrail Metric Analysis***

### ***Game Rounds Played***

In [ ]:
# Central tendency of the guardrail metric
rounds_control = control['sum_gamerounds']
rounds_treatment = treatment['sum_gamerounds']

print(f"Game Rounds Played")
print(f"{CONTROL}:   mean = {rounds_control.mean():.4f}, median = {rounds_control.median():.1f}")
print(f"{TREATMENT}: mean = {rounds_treatment.mean():.4f}, median = {rounds_treatment.median():.1f}")
print(f"Absolute difference in means: {rounds_treatment.mean() - rounds_control.mean():+.4f} rounds")

In [ ]:
# Welch t-test on the full sample
welch_t, welch_p = ttest_ind(rounds_treatment, rounds_control, equal_var=False)
print(f"Welch t-test (full sample)")
print(f"t = {welch_t:.4f}, p = {welch_p:.6f}")
print(f"Significant at alpha={ALPHA}: {welch_p < ALPHA}")

In [ ]:
# Mann-Whitney U test on the full sample
mw_u, mw_p = mannwhitneyu(rounds_treatment, rounds_control, alternative='two-sided')
print(f"Mann-Whitney U test (full sample)")
print(f"U = {mw_u:.1f}, p = {mw_p:.6f}")
print(f"Significant at alpha={ALPHA}: {mw_p < ALPHA}")

In [ ]:
# Welch t-test excluding the extreme record
rounds_control_clean = control_no_outlier['sum_gamerounds']
rounds_treatment_clean = treatment_no_outlier['sum_gamerounds']

welch_t_clean, welch_p_clean = ttest_ind(rounds_treatment_clean, rounds_control_clean, equal_var=False)
print(f"Welch t-test (outlier excluded)")
print(f"{CONTROL}:   mean = {rounds_control_clean.mean():.4f}")
print(f"{TREATMENT}: mean = {rounds_treatment_clean.mean():.4f}")
print(f"Absolute difference in means: {rounds_treatment_clean.mean() - rounds_control_clean.mean():+.4f} rounds")
print(f"t = {welch_t_clean:.4f}, p = {welch_p_clean:.6f}")

In [ ]:
# Compare the guardrail conclusion with and without the extreme record
print(f"Guardrail Sensitivity to the Extreme Record")
print(f"Mean difference including outlier: {rounds_treatment.mean() - rounds_control.mean():+.4f} rounds (p = {welch_p:.6f})")
print(f"Mean difference excluding outlier: {rounds_treatment_clean.mean() - rounds_control_clean.mean():+.4f} rounds (p = {welch_p_clean:.6f})")
print(f"The single largest record accounts for {abs((rounds_treatment.mean() - rounds_control.mean()) - (rounds_treatment_clean.mean() - rounds_control_clean.mean())):.4f} rounds of the gap")

## ***Permutation Test***

In [ ]:
# Exact permutation null for a binary metric
def permutation_test_proportion(metric, n_simulations=N_SIMULATIONS, seed=RANDOM_STATE):
    """Sample the permutation null of the rate difference.

    Under random reassignment the treatment successes follow a hypergeometric
    distribution, so the null can be sampled exactly without shuffling labels.
    """
    rng = np.random.default_rng(seed)
    successes = int(df[metric].sum())
    failures = len(df) - successes

    x_treatment_null = rng.hypergeometric(ngood=successes, nbad=failures,
                                          nsample=n_treatment, size=n_simulations)
    x_control_null = successes - x_treatment_null
    null_diffs = x_treatment_null / n_treatment - x_control_null / n_control

    observed = treatment[metric].mean() - control[metric].mean()
    p_value = (np.abs(null_diffs) >= abs(observed)).mean()
    return observed, null_diffs, p_value

In [ ]:
obs_diff_r7, null_diffs_r7, perm_p_r7 = permutation_test_proportion('retention_7')
print(f"Permutation Test - Day-7 Retention")
print(f"Observed difference: {obs_diff_r7*100:+.4f}pp")
print(f"Null distribution mean: {null_diffs_r7.mean()*100:+.6f}pp")
print(f"Null distribution std:  {null_diffs_r7.std()*100:.6f}pp")
print(f"Two-sided p-value: {perm_p_r7:.6f}")

In [ ]:
obs_diff_r1, null_diffs_r1, perm_p_r1 = permutation_test_proportion('retention_1')
print(f"Permutation Test - Day-1 Retention")
print(f"Observed difference: {obs_diff_r1*100:+.4f}pp")
print(f"Null distribution mean: {null_diffs_r1.mean()*100:+.6f}pp")
print(f"Null distribution std:  {null_diffs_r1.std()*100:.6f}pp")
print(f"Two-sided p-value: {perm_p_r1:.6f}")

## ***Bootstrap Analysis***

In [ ]:
# Nonparametric bootstrap for the difference in rates
def bootstrap_proportion_diff(metric, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_STATE):
    """Bootstrap the treatment-minus-control rate difference.

    Resampling a Bernoulli sample with replacement is equivalent to drawing the
    success count from Binomial(n, p_hat), which is used here directly.
    """
    rng = np.random.default_rng(seed)
    pc = control[metric].mean()
    pt = treatment[metric].mean()

    pc_star = rng.binomial(n_control, pc, n_bootstrap) / n_control
    pt_star = rng.binomial(n_treatment, pt, n_bootstrap) / n_treatment
    return pt_star - pc_star

In [ ]:
boot_r7 = bootstrap_proportion_diff('retention_7')
print(f"Bootstrap - Day-7 Retention ({N_BOOTSTRAP} resamples)")
print(f"Mean difference: {boot_r7.mean()*100:+.4f}pp")
print(f"95% percentile CI: [{np.percentile(boot_r7, 2.5)*100:+.4f}pp, {np.percentile(boot_r7, 97.5)*100:+.4f}pp]")
print(f"P(treatment worse than control): {(boot_r7 < 0).mean():.4f}")

In [ ]:
boot_r1 = bootstrap_proportion_diff('retention_1')
print(f"Bootstrap - Day-1 Retention ({N_BOOTSTRAP} resamples)")
print(f"Mean difference: {boot_r1.mean()*100:+.4f}pp")
print(f"95% percentile CI: [{np.percentile(boot_r1, 2.5)*100:+.4f}pp, {np.percentile(boot_r1, 97.5)*100:+.4f}pp]")
print(f"P(treatment worse than control): {(boot_r1 < 0).mean():.4f}")

In [ ]:
# Bootstrap the guardrail metric by resampling observed values
def bootstrap_mean_diff(values_control, values_treatment, n_bootstrap=5000, seed=RANDOM_STATE):
    """Bootstrap the difference in means for a continuous metric."""
    rng = np.random.default_rng(seed)
    vc = values_control.values
    vt = values_treatment.values
    diffs = np.empty(n_bootstrap)
    block = 500
    for start in range(0, n_bootstrap, block):
        size = min(block, n_bootstrap - start)
        ic = rng.integers(0, len(vc), size=(size, len(vc)))
        it = rng.integers(0, len(vt), size=(size, len(vt)))
        diffs[start:start+size] = vt[it].mean(axis=1) - vc[ic].mean(axis=1)
    return diffs

In [ ]:
boot_rounds = bootstrap_mean_diff(rounds_control, rounds_treatment)
print(f"Bootstrap - Game Rounds (5000 resamples, full sample)")
print(f"Mean difference: {boot_rounds.mean():+.4f} rounds")
print(f"95% percentile CI: [{np.percentile(boot_rounds, 2.5):+.4f}, {np.percentile(boot_rounds, 97.5):+.4f}]")
print(f"CI contains zero: {np.percentile(boot_rounds, 2.5) < 0 < np.percentile(boot_rounds, 97.5)}")

## ***Bayesian Analysis***

In [ ]:
# Beta-Binomial posterior comparison
def bayesian_proportion_test(metric, n_samples=200000, prior_alpha=1, prior_beta=1, seed=RANDOM_STATE):
    """Draw from Beta posteriors and summarize the treatment-vs-control comparison."""
    rng = np.random.default_rng(seed)
    xc, xt = int(control[metric].sum()), int(treatment[metric].sum())

    post_control = rng.beta(prior_alpha + xc, prior_beta + n_control - xc, n_samples)
    post_treatment = rng.beta(prior_alpha + xt, prior_beta + n_treatment - xt, n_samples)

    diff = post_treatment - post_control
    prob_treatment_better = (post_treatment > post_control).mean()
    expected_loss_treatment = np.maximum(post_control - post_treatment, 0).mean()
    expected_loss_control = np.maximum(post_treatment - post_control, 0).mean()

    return {
        'prob_treatment_better': prob_treatment_better,
        'expected_lift': diff.mean(),
        'relative_lift': (post_treatment / post_control - 1).mean(),
        'hdi_low': np.percentile(diff, 2.5),
        'hdi_high': np.percentile(diff, 97.5),
        'expected_loss_treatment': expected_loss_treatment,
        'expected_loss_control': expected_loss_control,
        'posterior_control': post_control,
        'posterior_treatment': post_treatment,
    }

In [ ]:
bayes_r7 = bayesian_proportion_test('retention_7')
print(f"Bayesian Analysis - Day-7 Retention (Beta(1,1) prior)")
print(f"P({TREATMENT} > {CONTROL}): {bayes_r7['prob_treatment_better']:.5f}")
print(f"Expected absolute lift: {bayes_r7['expected_lift']*100:+.4f}pp")
print(f"Expected relative lift: {bayes_r7['relative_lift']*100:+.4f}%")
print(f"95% credible interval: [{bayes_r7['hdi_low']*100:+.4f}pp, {bayes_r7['hdi_high']*100:+.4f}pp]")

In [ ]:
print(f"Expected Loss - Day-7 Retention")
print(f"Expected loss from shipping {TREATMENT}: {bayes_r7['expected_loss_treatment']*100:.6f}pp")
print(f"Expected loss from keeping {CONTROL}:   {bayes_r7['expected_loss_control']*100:.6f}pp")
print(f"Loss ratio: {bayes_r7['expected_loss_treatment']/bayes_r7['expected_loss_control']:.2f}x")

In [ ]:
bayes_r1 = bayesian_proportion_test('retention_1')
print(f"Bayesian Analysis - Day-1 Retention (Beta(1,1) prior)")
print(f"P({TREATMENT} > {CONTROL}): {bayes_r1['prob_treatment_better']:.5f}")
print(f"Expected absolute lift: {bayes_r1['expected_lift']*100:+.4f}pp")
print(f"Expected relative lift: {bayes_r1['relative_lift']*100:+.4f}%")
print(f"95% credible interval: [{bayes_r1['hdi_low']*100:+.4f}pp, {bayes_r1['hdi_high']*100:+.4f}pp]")

## ***Multiple Testing Correction***

In [ ]:
# Collect the p-values across all tested metrics
pvalue_table = pd.DataFrame([
    {'metric': 'retention_7', 'test': 'two-proportion z-test', 'pvalue': r7['z_pvalue']},
    {'metric': 'retention_1', 'test': 'two-proportion z-test', 'pvalue': r1['z_pvalue']},
    {'metric': 'sum_gamerounds', 'test': 'Welch t-test', 'pvalue': welch_p},
])
print(pvalue_table.to_string(index=False))

In [ ]:
# Apply Bonferroni and Benjamini-Hochberg corrections
bonf_reject, bonf_p, _, _ = multipletests(pvalue_table['pvalue'], alpha=ALPHA, method='bonferroni')
bh_reject, bh_p, _, _ = multipletests(pvalue_table['pvalue'], alpha=ALPHA, method='fdr_bh')

pvalue_table['bonferroni_p'] = bonf_p
pvalue_table['bonferroni_reject'] = bonf_reject
pvalue_table['bh_p'] = bh_p
pvalue_table['bh_reject'] = bh_reject

print(f"Multiple Testing Correction (alpha={ALPHA}, {len(pvalue_table)} tests)")
print(pvalue_table.round(6).to_string(index=False))

In [ ]:
# Confirm the primary metric survives correction
print(f"Primary metric ({PRIMARY_METRIC}) after correction")
primary_row = pvalue_table[pvalue_table['metric'] == PRIMARY_METRIC].iloc[0]
print(f"Raw p-value:         {primary_row['pvalue']:.6f}")
print(f"Bonferroni p-value:  {primary_row['bonferroni_p']:.6f}")
print(f"Benjamini-Hochberg:  {primary_row['bh_p']:.6f}")
print(f"Significant after Bonferroni: {primary_row['bonferroni_reject']}")

## ***Sequential Testing and Peeking Analysis***

In [ ]:
# Evolution of the primary metric p-value as the sample accumulates
def cumulative_pvalue_path(metric, n_checkpoints=20, seed=RANDOM_STATE):
    """Recompute the z-test at evenly spaced points in a randomized arrival order."""
    rng = np.random.default_rng(seed)
    shuffled = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    is_treatment = (shuffled['version'] == TREATMENT).values
    outcome = shuffled[metric].values

    cum_treatment_n = np.cumsum(is_treatment)
    cum_control_n = np.cumsum(~is_treatment)
    cum_treatment_x = np.cumsum(outcome * is_treatment)
    cum_control_x = np.cumsum(outcome * ~is_treatment)

    checkpoints = np.linspace(len(shuffled) / n_checkpoints, len(shuffled), n_checkpoints).astype(int) - 1
    rows = []
    for idx in checkpoints:
        nt, nc = cum_treatment_n[idx], cum_control_n[idx]
        xt, xc = cum_treatment_x[idx], cum_control_x[idx]
        if nt < 30 or nc < 30:
            continue
        z, p = proportions_ztest([xt, xc], [nt, nc])
        rows.append({'n_total': idx + 1, 'n_control': nc, 'n_treatment': nt,
                     'rate_control': xc / nc, 'rate_treatment': xt / nt,
                     'difference_pp': (xt / nt - xc / nc) * 100, 'pvalue': p})
    return pd.DataFrame(rows)

In [ ]:
sequential_df = cumulative_pvalue_path(PRIMARY_METRIC)
print(f"Cumulative Analysis - {PRIMARY_METRIC}")
print(sequential_df.round(6).to_string(index=False))

In [ ]:
# Point at which the primary metric first crosses the significance threshold
crossed = sequential_df[sequential_df['pvalue'] < ALPHA]
if len(crossed) > 0:
    first_cross = crossed.iloc[0]
    print(f"First checkpoint below alpha={ALPHA}: n_total = {int(first_cross['n_total'])}")
    print(f"Fraction of final sample: {first_cross['n_total']/len(df)*100:.1f}%")
    print(f"P-value at that checkpoint: {first_cross['pvalue']:.6f}")
    print(f"Checkpoints below alpha after first crossing: {len(crossed)} of {len(sequential_df)}")
else:
    print(f"No checkpoint fell below alpha={ALPHA}")

In [ ]:
# Type-I error inflation from repeated peeking, simulated under the null
def simulate_peeking_error(n_simulations=1000, n_checkpoints=20, seed=RANDOM_STATE):
    """Estimate the false positive rate of a fixed-horizon test evaluated repeatedly."""
    rng = np.random.default_rng(seed)
    outcome = df[PRIMARY_METRIC].values
    n_total = len(outcome)
    checkpoints = np.linspace(n_total / n_checkpoints, n_total, n_checkpoints).astype(int) - 1

    any_significant = 0
    final_significant = 0
    for _ in range(n_simulations):
        assignment = rng.permutation(np.concatenate([np.ones(n_treatment, dtype=bool),
                                                     np.zeros(n_control, dtype=bool)]))
        shuffled_outcome = rng.permutation(outcome)
        cum_nt = np.cumsum(assignment)
        cum_nc = np.cumsum(~assignment)
        cum_xt = np.cumsum(shuffled_outcome * assignment)
        cum_xc = np.cumsum(shuffled_outcome * ~assignment)

        hit = False
        for idx in checkpoints:
            nt, nc = cum_nt[idx], cum_nc[idx]
            if nt < 30 or nc < 30:
                continue
            pt, pc = cum_xt[idx] / nt, cum_xc[idx] / nc
            p_pool = (cum_xt[idx] + cum_xc[idx]) / (nt + nc)
            se = np.sqrt(p_pool * (1 - p_pool) * (1 / nt + 1 / nc))
            if se == 0:
                continue
            z = (pt - pc) / se
            p = 2 * (1 - stats.norm.cdf(abs(z)))
            if p < ALPHA:
                hit = True
            if idx == checkpoints[-1] and p < ALPHA:
                final_significant += 1
        if hit:
            any_significant += 1

    return any_significant / n_simulations, final_significant / n_simulations

In [ ]:
peek_error, fixed_error = simulate_peeking_error()
print(f"Peeking Simulation ({1000} null experiments, 20 checkpoints each)")
print(f"False positive rate testing only at the end: {fixed_error:.4f}")
print(f"False positive rate testing at every checkpoint: {peek_error:.4f}")
print(f"Inflation factor: {peek_error/fixed_error:.2f}x")
print(f"Nominal alpha: {ALPHA}")

In [ ]:
# Pocock-style adjusted boundary for the number of looks taken
n_looks = len(sequential_df)
pocock_alpha = 1 - (1 - ALPHA) ** (1 / n_looks)
print(f"Adjusted Boundary for Repeated Looks")
print(f"Number of looks: {n_looks}")
print(f"Per-look alpha to hold family-wise error at {ALPHA}: {pocock_alpha:.6f}")
print(f"Final p-value for {PRIMARY_METRIC}: {r7['z_pvalue']:.6f}")
print(f"Significant against the adjusted boundary: {r7['z_pvalue'] < pocock_alpha}")